# Day D5 — 집단 분석과 다중비교: 혼합효과, FWE, cluster, FDR, 순열, 검정력

설명은 학습 사이트(또는 원본 레슨)에서 읽고, 코드는 이 노트북에서 위에서 아래로 차례대로 실행하세요. 앞 셀이 만든 변수와 파일을 뒤 셀이 씁니다.

원본 레슨: [lessons/advanced/D-fmri/day-D5.md](https://github.com/evansuhohyun-927/neurodegeneration-bioinformatics/blob/main/lessons/advanced/D-fmri/day-D5.md)

In [ ]:
# Colab 기본 런타임에 없는 패키지 설치
%pip install -q nilearn

In [ ]:
# 레슨 코드가 쓰는 폴더를 미리 만든다 (Colab에서는 /content 아래에 생긴다)
import os
for d in ["data", "figures", "reports", "results", "../data", "../figures", "../reports"]:
    os.makedirs(d, exist_ok=True)

## 예시로 이해하기

### 예시 1 — 고정효과와 혼합효과: 같은 평균, 전혀 다른 결론

In [ ]:
import numpy as np
from scipy import stats

rng = np.random.default_rng(4)
n_sub = 12
true_pop_mean, sd_between = 0.2, 0.6        # 사람마다 진짜 효과가 크게 다르다
sd_within = 0.15                            # 1수준(한 사람 안) 추정 오차

subj_true = rng.normal(true_pop_mean, sd_between, n_sub)
beta = subj_true + rng.normal(0, sd_within, n_sub)      # 1수준에서 얻은 사람별 contrast 추정치
var_within = np.full(n_sub, sd_within ** 2)             # 1수준이 함께 알려 주는 분산

# 고정효과(FFX): 사람 안 오차만 쓴다 → '이 12명'에 대한 결론
w = 1 / var_within
ffx_mean = np.sum(w * beta) / w.sum()
ffx_se = np.sqrt(1 / w.sum())
print("FFX: 평균 %.2f, SE %.3f, z = %.1f" % (ffx_mean, ffx_se, ffx_mean / ffx_se))

# 혼합효과(요약 통계 방식, OLS): 사람별 추정치 12개로 one-sample t-test → '모집단'에 대한 결론
t, p = stats.ttest_1samp(beta, 0)
print("MFX(OLS): 평균 %.2f, SE %.3f, t(%d) = %.2f, p = %.3f"
      % (beta.mean(), beta.std(ddof=1) / np.sqrt(n_sub), n_sub - 1, t, p))
print("사람별 추정치:", np.round(beta, 2))

### 예시 2 — 신호가 있는 지도에서 보정 방법 다섯 가지 비교

In [ ]:
import numpy as np
from scipy import ndimage, stats
from statsmodels.stats.multitest import multipletests

rng = np.random.default_rng(8)
n_sub, shape = 16, (48, 48)                   # 피험자 16명, 2D '뇌' 48×48 = 2,304 voxel
sigma = 2.0                                   # 공간 smoothing (voxel 단위 σ)

def subject_maps(signal):
    maps = rng.normal(size=(n_sub, *shape))
    maps = np.array([ndimage.gaussian_filter(m, sigma) for m in maps])
    maps /= maps.std()
    return maps + signal

def tmap(maps):
    return maps.mean(0) / (maps.std(0, ddof=1) / np.sqrt(n_sub))

yy, xx = np.mgrid[0:48, 0:48]
signal = 1.2 * np.exp(-((xx - 14) ** 2 + (yy - 24) ** 2) / (2 * 4.0 ** 2))   # 왼쪽에만 진짜 효과
maps = subject_maps(signal)
t_obs = tmap(maps)
df, V = n_sub - 1, t_obs.size
truth = signal > 0.1

def report(name, thr):
    hit = t_obs > thr
    print(f"{name:22} 문턱 t>{thr:5.2f} | 검출 {hit.sum():4d} | 그중 신호 없는 곳 {np.sum(hit & ~truth):4d}")

report("보정 없음 p<0.05", stats.t.isf(0.05, df))
report("보정 없음 p<0.001", stats.t.isf(0.001, df))
report("Bonferroni FWE 0.05", stats.t.isf(0.05 / V, df))

# 순열(부호 뒤집기) 최대값 분포: 매번 지도 전체의 최대 t와 최대 cluster 크기를 기록
u_cluster = stats.t.isf(0.01, df)              # cluster를 만드는 1차 문턱
max_t, max_k = [], []
for _ in range(1000):
    flips = rng.choice([-1, 1], size=(n_sub, 1, 1))
    tp = tmap(maps * flips)
    max_t.append(tp.max())
    lab, nlab = ndimage.label(tp > u_cluster)
    max_k.append(np.bincount(lab.ravel())[1:].max() if nlab else 0)
report("순열 max-T FWE 0.05", np.quantile(max_t, 0.95))

k_crit = np.quantile(max_k, 0.95)
lab, nlab = ndimage.label(t_obs > u_cluster)
sizes = np.bincount(lab.ravel())[1:]
print(f"cluster 추론: 1차 문턱 t>{u_cluster:.2f}에서 cluster {nlab}개, 크기 {sorted(sizes.tolist(), reverse=True)[:5]}"
      f" | FWE 0.05 임계 크기 {k_crit:.0f} voxel → 살아남은 cluster {np.sum(sizes > k_crit)}개")

p = stats.t.sf(t_obs, df).ravel()
rej = multipletests(p, alpha=0.05, method="fdr_bh")[0].reshape(shape)
print(f"BH FDR 0.05            검출 {rej.sum():4d} | 그중 신호 없는 곳 {np.sum(rej & ~truth):4d}")

### 예시 3 — 미리 정한 ROI의 검정력 곡선

In [ ]:
import numpy as np
from scipy import stats

def power_one_sample(mu, sd, n, alpha=0.05):
    """미리 정한 ROI 하나의 평균 % 신호 변화에 대한 단측 one-sample t-test 검정력"""
    df = n - 1
    ncp = np.sqrt(n) * mu / sd                 # 비중심 모수
    return stats.nct.sf(stats.t.isf(alpha, df), df, ncp)

mu, sd = 0.5, 1.2                              # 가상 예비 자료: 평균 0.5%, 사람 간 SD 1.2%
for n in [10, 20, 30, 40, 50, 60]:
    print(f"N = {n:2d}: 검정력 {power_one_sample(mu, sd, n):.2f}")

need = next(n for n in range(5, 500) if power_one_sample(mu, sd, n) >= 0.8)
print("80%에 처음 닿는 N:", need)
# '만약'을 바꿔 보기: 효과가 20% 작다면?
need_small = next(n for n in range(5, 500) if power_one_sample(0.8 * mu, sd, n) >= 0.8)
print("효과가 20% 작으면 필요한 N:", need_small)

# 자기 점검: 시뮬레이션으로 같은 값을 다시 얻는지
rng = np.random.default_rng(0)
sims = rng.normal(mu, sd, size=(20000, need))
p = stats.ttest_1samp(sims, 0, axis=1, alternative="greater").pvalue
print("시뮬레이션 검정력(N=%d): %.2f" % (need, np.mean(p < 0.05)))

### 참고 — 실제 집단 분석·순열 검정 도구의 명령 형태 (FSL 부분은 실행 확인 안 됨)

In [ ]:
# 내 컴퓨터용 명령입니다. Colab에서는 의미가 없어 실행하지 않도록 주석 처리했습니다.
# # FSL randomise: 사람별 contrast 지도를 4D로 쌓은 파일에 one-sample t, 부호 뒤집기 5,000번, TFCE.
# randomise -i all_subjects_cope.nii.gz -o grp -1 -T -n 5000 -m group_mask.nii.gz

In [ ]:
# nilearn 2수준 모형. nilearn 0.14.1에서 가상 contrast 지도 8개로 실행 확인(n_perm만 줄여서). FSL randomise는 FSL 미설치로 실행 확인 안 됨.
from nilearn.glm.second_level import SecondLevelModel, non_parametric_inference
import pandas as pd
design = pd.DataFrame({"intercept": [1] * len(cmap_files)})
z = SecondLevelModel().fit(cmap_files, design_matrix=design).compute_contrast("intercept", output_type="z_score")
neg_log_p = non_parametric_inference(cmap_files, design_matrix=design, n_perm=5000)